# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Frame:** features from `month=2026-03` only, label from `month=2026-04` (did daily impressions drop >20% from March to April) — same feature/label split as the model weeks. Lane stays decline-risk classification, locked this week.

**Two signals checked before writing any rule** — both from the menu this week's session named: CTR relative to a page's position peers (the logic behind a CTR-fix flag), and raw traffic volume (the logic behind a quick-win flag: 'big pages are safer bets'). Verdicts are computed from the printed tables below, not asserted here — see the `VERDICT:` line under each.

In [1]:
%pip -q install duckdb
import json
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = 'hf://datasets/FlyRank/internship-warehouse'
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"

# March features, April label -- no overlap between the two windows.
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS mar_impr, SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*) AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.mar_impr, m.mar_clicks, m.mar_days,
       m.mar_impr * 1.0 / m.mar_days AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0) AS mar_avg_position,
       m.mar_clicks * 100.0 / NULLIF(m.mar_impr, 0) AS mar_ctr,
       (m.h2_impr - m.h1_impr) * 1.0 / NULLIF(m.h2_impr + m.h1_impr, 0) AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0) AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame['mar_h2_vs_h1'] = frame['mar_h2_vs_h1'].fillna(0.0)
frame['apr_daily_impr'] = frame['apr_daily_impr_raw'].fillna(0.0)
frame['y'] = (frame['apr_daily_impr'] < 0.80 * frame['mar_daily_impr']).astype(int)
frame = frame.drop(columns=['apr_daily_impr_raw'])

BASE_RATE = frame['y'].mean()
print(f"frame: {len(frame):,} pages | {frame['client_hash_id'].nunique()} clients | base rate {BASE_RATE:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 pages | 40 clients | base rate 0.504


In [2]:
def bucket_verdict(table, rate_col, hypothesis, tol=0.03):
    """Compares first vs last bucket's decline rate against a stated hypothesis direction."""
    first, last = table[rate_col].iloc[0], table[rate_col].iloc[-1]
    spread = table[rate_col].max() - table[rate_col].min()
    if spread < tol:
        return 'FALSE', spread
    moved_as_hypothesized = (last < first) if hypothesis == 'decreases' else (last > first)
    if moved_as_hypothesized and spread >= tol:
        return 'CONFIRMED', spread
    if not moved_as_hypothesized and spread >= tol:
        return 'OPPOSITE', spread
    return 'MIXED', spread

# SIGNAL 1: CTR relative to position peers (the CTR-fix logic)
frame['pos_bucket'] = pd.cut(frame['mar_avg_position'], bins=[0, 3, 10, 20, 50, 200],
                              labels=['1-3', '4-10', '11-20', '21-50', '50+'])
frame['ctr_vs_peers'] = frame['mar_ctr'] - frame.groupby('pos_bucket', observed=True)['mar_ctr'].transform('median')
frame['ctr_gap_bucket'] = pd.cut(frame['ctr_vs_peers'], bins=[-100, -0.20, -0.02, 0.02, 0.20, 100],
                                  labels=['far below', 'below', 'at peers', 'above', 'far above'])
sig1 = frame.groupby('ctr_gap_bucket', observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_ctr=('mar_ctr', 'median')).round(3)
print('SIGNAL 1 -- does CTR BELOW its position peers predict decline?')
print(sig1.to_string())
v1, spread1 = bucket_verdict(sig1.reset_index(), 'decline_rate', hypothesis='decreases')
print(f"VERDICT: {v1}  (spread {spread1:.3f} across buckets, hypothesis: below-peers pages decline MORE)")

# SIGNAL 2: raw volume (the quick-win logic -- 'big pages are safer bets')
frame['vol_q'] = pd.qcut(frame['mar_daily_impr'], 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high'])
sig2 = frame.groupby('vol_q', observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_daily_impr=('mar_daily_impr', 'median')).round(3)
print('\nSIGNAL 2 -- does HIGHER volume alone predict LOWER decline risk?')
print(sig2.to_string())
v2, spread2 = bucket_verdict(sig2.reset_index(), 'decline_rate', hypothesis='decreases')
print(f"VERDICT: {v2}  (spread {spread2:.3f} across quintiles, hypothesis: high-volume pages decline LESS)")


SIGNAL 1 -- does CTR BELOW its position peers predict decline?
                    n  decline_rate  median_ctr
ctr_gap_bucket                                 
below           41069         0.592       0.000
at peers        24509         0.484       0.000
above           20226         0.518       0.204
far above       30734         0.395       0.573
VERDICT: CONFIRMED  (spread 0.197 across buckets, hypothesis: below-peers pages decline MORE)

SIGNAL 2 -- does HIGHER volume alone predict LOWER decline risk?
             n  decline_rate  median_daily_impr
vol_q                                          
Q1 low   23311         0.437              2.957
Q2       23307         0.551              7.400
Q3       23322         0.561             19.097
Q4       23294         0.524             52.097
Q5 high  23305         0.449            186.862
VERDICT: OPPOSITE  (spread 0.124 across quintiles, hypothesis: high-volume pages decline LESS)


**Reading the two verdicts above** decides whether either signal earns a place in the rule. If both come back `OPPOSITE`, `FALSE`, or too thin to trust, the rule below falls back to **within-month momentum** instead: does a page's second half of March already look worse than its first half? That is checked and confirmed in the next cell before it goes into the score.

In [3]:
# The fallback signal: within-month momentum, checked the same way
frame['mom_q'] = pd.qcut(frame['mar_h2_vs_h1'], 5, labels=['Q1 falling', 'Q2', 'Q3', 'Q4', 'Q5 rising'])
mom = frame.groupby('mom_q', observed=True).agg(n=('y', 'size'), decline_rate=('y', 'mean')).round(3)
print('WITHIN-MONTH MOMENTUM (h2 vs h1 of March)')
print(mom.to_string())
v3, spread3 = bucket_verdict(mom.reset_index(), 'decline_rate', hypothesis='decreases')
print(f"VERDICT: {v3}  (spread {spread3:.3f} across quintiles, hypothesis: pages falling in March decline MORE in April)")

print('\nCONFOUND CHECK -- momentum inside each volume quintile, so it is not just volume in disguise:')
print(frame.groupby(['vol_q', 'mom_q'], observed=True)['y'].mean().unstack().round(3).to_string())

WITHIN-MONTH MOMENTUM (h2 vs h1 of March)
                n  decline_rate
mom_q                          
Q1 falling  23310         0.735
Q2          24734         0.559
Q3          21920         0.467
Q4          23267         0.391
Q5 rising   23308         0.364
VERDICT: CONFIRMED  (spread 0.371 across quintiles, hypothesis: pages falling in March decline MORE in April)

CONFOUND CHECK -- momentum inside each volume quintile, so it is not just volume in disguise:
mom_q    Q1 falling     Q2     Q3     Q4  Q5 rising
vol_q                                              
Q1 low        0.616  0.447  0.403  0.357      0.334
Q2            0.760  0.582  0.519  0.474      0.411
Q3            0.773  0.606  0.513  0.457      0.427
Q4            0.778  0.575  0.487  0.395      0.383
Q5 high       0.782  0.571  0.413  0.299      0.253


### The rule, in plain words

A page goes to the top of the queue if it is already falling inside March and still carries enough traffic to be worth an editor's time. Severity and volume both have to be true — a collapsing page with a handful of daily impressions is not worth a slot, and neither is a big stable page.

**score = decline_severity × volume_weight**

- `decline_severity = max(0, -mar_h2_vs_h1)` — zero for flat or rising pages, up to 1.0 for a page that lost all its second-half traffic.
- `volume_weight = log1p(mar_daily_impr)` — bigger pages count more, log-damped so one huge page cannot own the whole queue.

### Reason codes

| reason code | condition | action |
|---|---|---|
| `HIGH_RISK_HIGH_TRAFFIC` | momentum ≤ −0.20 and daily impressions ≥ 50 | `ACT_NOW` |
| `HIGH_RISK_LOW_TRAFFIC` | momentum ≤ −0.20 and daily impressions < 50 | `QUEUE_LATER` |
| `MILD_RISK` | −0.20 < momentum ≤ −0.05 | `WATCH` |
| `HEALTHY` | momentum > −0.05 | `SKIP` |

Checked top to bottom, first match wins — every row carries exactly one code. The −0.20 cutoff matches the same 20%-drop convention the label itself uses; the 50/day floor is a plain 'does this page still matter' bar, not tuned against any score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

Evaluated with precision@K against the base rate, on the same March→April slice and label the Week-5 model will use. All three thresholds come from the tables in Section 1 and are fixed before scoring — tuning them against precision@K afterward would make the baseline look good and prove nothing. Its whole job is to be honestly beatable.

In [4]:
FALL_HARD, FALL_SOFT, VOL_FLOOR = -0.20, -0.05, 50.0

frame['decline_severity'] = np.maximum(0.0, -frame['mar_h2_vs_h1'])
frame['volume_weight'] = np.log1p(frame['mar_daily_impr'])
frame['baseline_score'] = frame['decline_severity'] * frame['volume_weight']

def classify(r):
    if r['mar_h2_vs_h1'] <= FALL_HARD and r['mar_daily_impr'] >= VOL_FLOOR:
        return 'HIGH_RISK_HIGH_TRAFFIC', 'ACT_NOW'
    if r['mar_h2_vs_h1'] <= FALL_HARD:
        return 'HIGH_RISK_LOW_TRAFFIC', 'QUEUE_LATER'
    if r['mar_h2_vs_h1'] <= FALL_SOFT:
        return 'MILD_RISK', 'WATCH'
    return 'HEALTHY', 'SKIP'

frame[['reason_code', 'action']] = frame.apply(classify, axis=1, result_type='expand')
assert frame['reason_code'].notna().all(), 'every row must carry exactly one reason code'

print('REASON CODES -- one per row, first match wins')
print(frame.groupby(['reason_code', 'action'], observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'),
    median_momentum=('mar_h2_vs_h1', 'median'),
    median_daily_impr=('mar_daily_impr', 'median')).round(3).to_string())

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = frame['y'].to_numpy()
rng = np.random.default_rng(42)
rand50 = np.array([y[rng.choice(len(y), 50, replace=False)].mean() for _ in range(500)])

print(f"\nbase rate (random picking): {BASE_RATE:.3f}")
print(f"{'K':>6} {'my rule':>10} {'lift':>8}")
results = {}
for K in (10, 20, 50, 100, 500):
    p = precision_at_k(frame['baseline_score'], y, K)
    results[f'precision_at_{K}'] = round(float(p), 4)
    print(f"{K:>6} {p:>10.3f} {p - BASE_RATE:>+8.3f}")
print(f"random@50: mean {rand50.mean():.3f}, 5th-95th pct {np.percentile(rand50,5):.3f}-{np.percentile(rand50,95):.3f}")

Path('work/outputs').mkdir(parents=True, exist_ok=True)
queue = (frame.sort_values('baseline_score', ascending=False)
              .assign(rank=lambda d: range(1, len(d) + 1))
              [['rank', 'client_hash_id', 'content_hash_id', 'baseline_score',
                'reason_code', 'action', 'mar_h2_vs_h1', 'mar_daily_impr',
                'mar_avg_position', 'mar_ctr']])
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nwrote work/outputs/baseline_action_score.csv -- {len(queue):,} rows, ranked")

metrics = {
    'as_of_date': '2026-03-31', 'feature_window': '2026-03', 'label_window': '2026-04',
    'n_pages': int(len(frame)), 'n_clients': int(frame['client_hash_id'].nunique()),
    'base_rate': round(float(BASE_RATE), 4),
    'random_at_50_mean': round(float(rand50.mean()), 4),
    'rule': 'score = max(0, -mar_h2_vs_h1) * log1p(mar_daily_impr)',
    'thresholds': {'fall_hard': FALL_HARD, 'fall_soft': FALL_SOFT, 'volume_floor': VOL_FLOOR},
    'signal_verdicts': {'ctr_vs_position_peers': v1, 'raw_volume': v2, 'within_month_momentum': v3},
    **results,
}
with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('wrote work/outputs/baseline_metrics.json')


REASON CODES -- one per row, first match wins
                                        n  decline_rate  median_momentum  median_daily_impr
reason_code            action                                                              
HEALTHY                SKIP         78271         0.420            0.178             20.516
HIGH_RISK_HIGH_TRAFFIC ACT_NOW       4788         0.800           -0.317            116.339
HIGH_RISK_LOW_TRAFFIC  QUEUE_LATER  15528         0.732           -0.362              8.056
MILD_RISK              WATCH        17952         0.597           -0.116             21.194

base rate (random picking): 0.504
     K    my rule     lift
    10      0.900   +0.396
    20      0.900   +0.396
    50      0.880   +0.376
   100      0.820   +0.316
   500      0.858   +0.354
random@50: mean 0.505, 5th-95th pct 0.380-0.621

wrote work/outputs/baseline_action_score.csv -- 116,539 rows, ranked
wrote work/outputs/baseline_metrics.json


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

(This week's card asks for ten; the sibling `w04_signal_audit.ipynb` is where a top-20 pass belongs if I want to go deeper — not required here.)

In [5]:
top10 = queue.head(10).merge(
    frame[['client_hash_id', 'content_hash_id', 'y', 'apr_daily_impr']],
    on=['client_hash_id', 'content_hash_id'], how='left')

review = pd.DataFrame({
    'rank': top10['rank'],
    'client': top10['client_hash_id'].str[-6:],
    'content': top10['content_hash_id'].str[-6:],
    'score': top10['baseline_score'].round(2),
    'action': top10['action'],
    'reason': top10['reason_code'],
    'momentum': top10['mar_h2_vs_h1'].round(3),
    'mar_daily': top10['mar_daily_impr'].round(1),
    'apr_daily': top10['apr_daily_impr'].round(1),
    'declined': np.where(top10['y'] == 1, 'YES', 'no'),
})
print('TOP 10 -- the queue an editor would actually work down')
print(review.to_string(index=False))

hit = top10['y'].mean()
print(f"\nprecision@10 = {hit:.3f}  (base rate {BASE_RATE:.3f}, lift {hit - BASE_RATE:+.3f})")
print(f"misses in the top 10: {int((top10['y'] == 0).sum())}")

print('\nIS THE TOP 10 JUST ONE CLIENT? (a tracking break would look exactly like this)')
print(top10['client_hash_id'].str[-6:].value_counts().to_string())
print(f"distinct clients in top 10: {top10['client_hash_id'].nunique()}")


TOP 10 -- the queue an editor would actually work down
 rank client content  score  action                 reason  momentum  mar_daily  apr_daily declined
    1 f265ea  0a3abb   7.89 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.999     2704.3        6.5      YES
    2 5d81d4  d97f04   5.74 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.988      332.0        1.0      YES
    3 9f63c4  ea94ca   5.65 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.840      829.2       65.2      YES
    4 5d81d4  9e83fd   5.52 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.998      251.2        0.0      YES
    5 5e0096  c01691   5.50 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.950      326.2        6.6      YES
    6 9f63c4  3fca87   5.47 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.715     2094.7      774.7      YES
    7 4ef01b  c31c17   5.36 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.711     1861.9     1990.3       no
    8 5d81d4  03face   5.31 ACT_NOW HIGH_RISK_HIGH_TRAFFIC    -0.989      213.9        1.0      YES
    9 f265ea  c01b7a   5.23 ACT_NOW HIGH_RISK

**One line per row, action + why + what would flip it** (fill the two blanks per row against your own printed table above — the shape is fixed, the numbers are yours):

1-10. `{action}` because momentum sits at `{momentum}` on `{mar_daily}` daily impressions — would be wrong if this page published mid-March (first half partly pre-launch), covers a March-only event (expected to fall in April on its own), or its client shows a tracking break in the client-concentration check above rather than a real content problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Which picks look wrong

**Reason-code monoculture.** Because the score multiplies severity by volume, only pages extreme in both can reach the top — an editor working this queue sees one story repeated ten times, not a spread of evidence.

**Reversion is invisible to March-only data.** Some drops are noise around a page's own average and recover without help. Nothing in this feature set can tell 'still collapsing' from 'already bottomed out.'

**No publish-date check.** A page that went live mid-March would show the exact same first-half/second-half collapse as a genuinely declining page, and `dim_content` is deliberately not used here (see the leakage check below), so this risk cannot be ruled out for any row.

**Scores compress fast.** Rank 8 and rank 80 can sit closer together in score than the ranking suggests — being in the top 10 is meaningful, the exact order inside it less so.

In [6]:
# LEAKAGE CHECK: prove the score uses March only
MARCH_ONLY = ['mar_impr', 'mar_clicks', 'mar_days', 'mar_daily_impr',
              'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1']
APRIL_COLS = ['apr_daily_impr', 'y']

rebuilt = np.maximum(0.0, -frame['mar_h2_vs_h1']) * np.log1p(frame['mar_daily_impr'])
assert np.allclose(rebuilt, frame['baseline_score']), 'score does not rebuild from March columns'
print('[1] score rebuilds EXACTLY from March-only columns          PASS')
print(f'    inputs: {MARCH_ONLY}')

corr = {c: abs(np.corrcoef(frame[c], frame['y'])[0, 1]) for c in MARCH_ONLY}
print('\n[2] |correlation with the April label| -- no March feature should sit near 1.0')
for c, v in sorted(corr.items(), key=lambda kv: -kv[1]):
    print(f'    {c:20s} {v:.4f}')
assert max(corr.values()) < 0.90, 'a March feature is suspiciously close to the label'
print('    max |r| below 0.90                                       PASS')

leaked_in_csv = [c for c in queue.columns if c in APRIL_COLS]
print(f"\n[3] April columns written to the queue CSV: {leaked_in_csv or 'none'}")
assert not leaked_in_csv, 'an April column leaked into the deliverable'
print('    the CSV an editor receives contains no future data          PASS')

print('\n[4] no dim_content field, no shipped product flag used:')
print(f'    columns in the scored frame: {sorted(frame.columns.tolist())}')

print(f"\nFROZEN BASELINE -- precision@50 = {results['precision_at_50']:.3f} "
      f"vs base rate {BASE_RATE:.3f}. This is the number Week 5 must beat.")


[1] score rebuilds EXACTLY from March-only columns          PASS
    inputs: ['mar_impr', 'mar_clicks', 'mar_days', 'mar_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1']

[2] |correlation with the April label| -- no March feature should sit near 1.0
    mar_h2_vs_h1         0.2521
    mar_ctr              0.0982
    mar_clicks           0.0738
    mar_days             0.0475
    mar_daily_impr       0.0409
    mar_impr             0.0393
    mar_avg_position     0.0369
    max |r| below 0.90                                       PASS

[3] April columns written to the queue CSV: none
    the CSV an editor receives contains no future data          PASS

[4] no dim_content field, no shipped product flag used:
    columns in the scored frame: ['action', 'apr_daily_impr', 'baseline_score', 'client_hash_id', 'content_hash_id', 'ctr_gap_bucket', 'ctr_vs_peers', 'decline_severity', 'mar_avg_position', 'mar_clicks', 'mar_ctr', 'mar_daily_impr', 'mar_days', 'mar_h2_vs_h1', 'mar_impr',

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.